# Lesson 17 | What is external memory?

On-chip **Random-Access Memory (RAM)** is close to logic but limited in capacity. A real **connectome** synapse store quickly outgrows it, so we need:

> **large storage outside the Field-Programmable Gate Array (FPGA) fabric.**

Primary new concept: **Double Data Rate Synchronous Dynamic Random-Access Memory (DDR SDRAM) has its own access latency, and contiguous bursts cost differently from scattered random accesses.**

## 1. Concept ledger

**Already known:** memory hierarchy, latency, bandwidth, synapse records.

**New today:**
- **Double Data Rate Synchronous Dynamic Random-Access Memory (DDR SDRAM)**;
- **word**: in this lesson, one individually addressable data unit;
- **burst**: grouping several adjacent words into one contiguous batch so they can share fixed startup cost;
- sequential access and random access.

**Preview only:** the next lesson introduces a standard on-chip communication protocol commonly used above this memory path.

## 2. Why not keep every synapse on-chip?

```mermaid
flowchart LR
  ENG["synapse engine"] --> CTRL["platform memory controller"]
  CTRL --> DDR["external DDR memory"]
```

DDR provides capacity at the cost of a longer and more complex access path. The project uses the platform-provided memory controller rather than asking students to implement the low-level DDR physical interface.

## 3. Why are contiguous accesses often friendlier?

Many accesses carry a fixed startup cost. Bursts can amortize that cost across adjacent data, while random access may require more frequent restarts of a request or transfer.

## 4. A teaching burst model

Treat addresses as word indices. Continue a burst only when `current == previous + 1` and the burst is not full. This models access patterns, not a DDR controller.

## 5. Run: contiguous versus random-like addresses

Predict the burst count for eight sequential words with maximum burst length four.

In [ ]:
def estimate_bursts(addresses, max_burst_words):
    if not addresses:
        return 0
    bursts = 1
    run_length = 1
    for previous, current in zip(addresses, addresses[1:]):
        if current == previous + 1 and run_length < max_burst_words:
            run_length += 1
        else:
            bursts += 1
            run_length = 1
    return bursts

sequential = list(range(8))
random_like = [0, 9, 2, 14, 7, 20, 1, 30]

print("sequential bursts:", estimate_bursts(sequential, 4))
print("random-like bursts:", estimate_bursts(random_like, 4))


## 6. Observe

Eight contiguous addresses form two four-word bursts. The random-like sequence starts a new burst almost every time. Equal byte counts can have different transfer cost.

## 7. Why integrity before performance?

DDR hello-world first writes known values and reads them back reliably. Only after integrity is stable does bandwidth benchmarking mean anything.

## 8. Try It

Change the sequential list to `list(range(10))` with max burst four, then change max burst to eight. Predict first.

## 9. Exercise

[Lesson 17 exercise: estimate burst cost for sequential/random access](../../exercises/en/17_external_memory_ddr.ipynb)

## 10. AI Task

Ask an AI why reading the same 1 KB sequentially versus randomly can cost differently. Check that it does not claim random access is functionally impossible.

## 11. Human Check

Explain why DDR solves capacity while adding access cost, why bursts help contiguous data, how access pattern affects effective bandwidth, and why the course uses a platform-provided memory controller rather than asking students to implement the low-level DDR physical interface.

## 12. Engineering Handoff

Maps to `RMD-014`: use the platform-provided memory controller for reliable DDR read/write plus integrity testing.

## 13. Project Trace

- Lesson: `LSN-017`
- Mapping: `RMD-014`
- First proof: DDR read/write integrity
- Performance preview: sequential vs random vs burst

## 14. Exit Ticket

You can explain why external DDR provides capacity with different access cost and estimate burst count from a simple address sequence.